# Memory Bank Setup

We explore two strategies for building a reference memory bank using DINOv2 patch embeddings:

1. **Per-class centroids** — top-`k` images per class closest to its class centroid → `k × num_classes` reference images, saved as CSV and copied to disk.
2. **Global centroid** — top-`k` images across the entire dataset closest to the global centroid → `k` reference images total.

In [ ]:
import sys
sys.path.insert(0, "..")

import shutil
import csv
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as v2
from torch.utils.data import DataLoader
from tqdm import tqdm

from utils.datasets import SimpleDataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

feature_extractor = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
feature_extractor.eval().to(device)


def extract_patch_embedding(image_tensor):
    """Mean-pool DINOv2 patch tokens → (batch, embed_dim) L2-normalised vector."""
    with torch.inference_mode():
        tokens = feature_extractor.get_intermediate_layers(image_tensor.to(device), n=1)[0]
    return F.normalize(tokens.mean(dim=1), dim=-1).cpu()

In [ ]:
DATASET_ROOT  = "/Users/kaygijzen/Downloads/Stanford_Cars_dataset"
IMAGES_SUBDIR = "train"
BATCH_SIZE    = 32
K             = 1  # reference images to keep (per class for method 1, total for method 2)

dataset = SimpleDataset(
    root=DATASET_ROOT,
    images_subdir=IMAGES_SUBDIR,
    transform=v2.Compose([
        v2.Resize(256, antialias=True),
        v2.CenterCrop(224),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    include_metadata=True,
)
classes = dataset.classes()
print(f"Classes : {len(classes)}")
print(f"Images  : {len(dataset)}")

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# --- Extract embeddings for all training images ---
all_embeddings = []  # list of (class_idx, path, embedding tensor)

for imgs, labels, meta in tqdm(loader, desc="Extracting embeddings"):
    embeddings = extract_patch_embedding(imgs)  # (batch, embed_dim)
    for i in range(len(labels)):
        all_embeddings.append({
            "class_idx": labels[i].item(),
            "class_name": classes[labels[i].item()],
            "path": meta["path"][i],
            "embedding": embeddings[i],
        })

print(f"Extracted {len(all_embeddings)} embeddings")

Extracting embeddings:   0%|                                                                                                   | 0/255 [00:00<?, ?it/s]/Users/kaygijzen/Downloads/oasic/occlusion_utils-main/env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Extracting embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 255/255 [04:29<00:00,  1.06s/it]

Extracted 8144 embeddings


## Method 1 — Per-class centroids (`k × num_classes` reference images)

For each class, compute the mean embedding (centroid) over all its training images, then select the `k` images with the highest cosine similarity to that centroid.

In [30]:
class_items = defaultdict(list)
for item in all_embeddings:
    class_items[item["class_name"]].append(item)

memory_bank = {}  # class_name → [path, ...]

for class_name, items in tqdm(class_items.items(), desc="Per-class centroids"):
    embs     = torch.stack([item["embedding"] for item in items])
    centroid = F.normalize(embs.mean(dim=0), dim=-1)
    sims     = (embs @ centroid).tolist()
    ranked   = sorted(zip(sims, items), key=lambda x: -x[0])
    memory_bank[class_name] = [item["path"] for _, item in ranked[:K]]

print(f"Memory bank: {len(memory_bank)} classes × {K} image(s) each")

Per-class centroids: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:00<00:00, 5333.22it/s]

Memory bank: 196 classes × 1 image(s) each


In [ ]:
csv_path = Path("memory_bank.csv")

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_name"] + [f"{i+1}_closest_image_path" for i in range(K)])
    for class_name in classes:
        if class_name in memory_bank:
            writer.writerow([class_name] + memory_bank[class_name])

print(f"Saved: {csv_path.resolve()}")

In [ ]:
reference_bank_path = Path("examples/data/reference_images")

for class_name, paths in memory_bank.items():
    class_dir = reference_bank_path / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    for rank, src in enumerate(paths, 1):
        dst = class_dir / f"{rank}_{Path(src).name}"
        shutil.copy2(src, dst)

print(f"Copied {sum(len(v) for v in memory_bank.values())} images to {reference_bank_path.resolve()}")

Copied 196 images to /Users/kaygijzen/Downloads/oasic/occlusion_utils-main/examples/data/reference_images_1


## Method 2 — Global centroid (`k` reference images total)

Compute a single centroid across all training images and select the `k` images with the highest cosine similarity to it, regardless of class.

In [42]:
k_global = 5  # top-k images closest to global centroid to save as reference bank

global_centroid = F.normalize(
    torch.stack([e["embedding"] for e in all_embeddings]).mean(dim=0), dim=-1
)

global_top_k = sorted(
    all_embeddings, key=lambda x: -(x["embedding"] @ global_centroid).item()
)[:k_global]

for rank, item in enumerate(global_top_k, 1):
    print(f"{rank}. [{item['class_name']}] {item['path']}")

1. [Chrysler PT Cruiser Convertible 2008] /Users/kaygijzen/Downloads/Stanford_Cars_dataset/train/Chrysler PT Cruiser Convertible 2008/006532.jpg
2. [Dodge Charger Sedan 2012] /Users/kaygijzen/Downloads/Stanford_Cars_dataset/train/Dodge Charger Sedan 2012/007818.jpg
3. [BMW X6 SUV 2012] /Users/kaygijzen/Downloads/Stanford_Cars_dataset/train/BMW X6 SUV 2012/002643.jpg
4. [Chevrolet Malibu Hybrid Sedan 2010] /Users/kaygijzen/Downloads/Stanford_Cars_dataset/train/Chevrolet Malibu Hybrid Sedan 2010/005382.jpg
5. [Daewoo Nubira Wagon 2002] /Users/kaygijzen/Downloads/Stanford_Cars_dataset/train/Daewoo Nubira Wagon 2002/006641.jpg


In [45]:
reference_bank_path = Path("examples/data/reference_images")

for _, item in enumerate(global_top_k):
    class_dir = reference_bank_path / "global_top_k"
    class_dir.mkdir(parents=True, exist_ok=True)
    dst = class_dir / f"{item['class_name']}_{Path(item['path']).name}"
    shutil.copy2(item['path'], dst)

print(f"Copied {len(global_top_k)} images to {reference_bank_path.resolve()}")

Copied 5 images to /Users/kaygijzen/Downloads/oasic/occlusion_utils-main/examples/data/reference_images
